# Bulwark: big dataset with Lc0 labels (CPU only, no Stockfish)

**This notebook only builds data. No GPU, no training.** It downloads Lc0 training games (several in parallel),
converts them to positions with Lc0's own searched win/draw/loss labels plus the game result, and writes training
shards (`train/`, `val/`, `test/`). Target: **300M positions per session** (5 sessions = 1.5B).

**Settings (right-hand panel):** Accelerator **None**, Internet **On**, and the `bulwark` dataset (latest
`bulwark.zip`) added as an input.

**Run it:** Save Version -> **Save & Run All (Commit)**. Run the 5 notebooks (`SESSION` 0-4) at the same time.

**Checkpointing:** progress is saved after every ~2M positions; the run stops starting new work at 11 h and keeps
output under 18 GB. If the last cell says **NOT DONE**: Add Input -> Your Work -> this notebook's previous output,
then Save & Run All again. Each round writes only new files: download the output of **every** round.


In [ ]:
# ---- Parameters -------------------------------------------------------------
SESSION = 0             # 0..4: a different value in each copy of this notebook
NUM_SESSIONS = 5        # how many copies run in total (keep 5 unless you change all of them)
TARGET = 300_000_000    # positions for this session (5 sessions x 300M = 1.5B)
DEADLINE_H = 11.0       # stop starting new work after this many hours (Kaggle CPU limit is 12 h)

import os, time, glob, json, subprocess, shutil
START = time.time()
WORK = "/kaggle/working"     # saved when the version finishes
TMP = "/kaggle/tmp"          # scratch: code, tools, downloads (not saved)
os.makedirs(TMP, exist_ok=True)
print("CPUs:", os.cpu_count())

In [ ]:
# ---- Setup: code, converter, 3-5 piece tablebases (all in /kaggle/tmp) ----
def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

bundles = glob.glob("/kaggle/input/**/bulwark.bundle", recursive=True)
if not bundles:   # Kaggle normally unpacks an uploaded zip; if it kept the zip, unpack it here
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        sh(f"unzip -o -q {z} -d {TMP}/bundle")
    bundles = glob.glob(f"{TMP}/bundle/**/bulwark.bundle", recursive=True)
assert bundles, "add the dataset made from bulwark.zip as an input"
CODE = f"{TMP}/bulwark"
if not os.path.isdir(CODE):
    sh(f"git clone -q {bundles[0]} {CODE}")
sh(f"cd {CODE} && git log --oneline -1")
sh(f"cd {CODE} && cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j{os.cpu_count()} --target lc0_convert > /dev/null")

TB = f"{TMP}/syzygy"
if len(glob.glob(f"{TB}/*.rtbw")) < 100:
    os.makedirs(TB, exist_ok=True)
    for kind in ("3-4-5-wdl", "3-4-5-dtz"):
        url = f"https://tablebase.lichess.ovh/tables/standard/{kind}/"
        sh(f"curl -sfL {url} | grep -o 'href=\"[^\"]*rtb[wz]\"' | sed 's/href=\"//;s/\"//' | "
           f"sed 's#^#{url}#' | xargs -P 8 -n 1 curl -sfL -O --output-dir {TB}")
n = len(glob.glob(f"{TB}/*.rtbw")); print("tablebase WDL files:", n); assert n >= 100, "tablebase download failed"

In [ ]:
# ---- Build data (continues automatically from an earlier version's output added as input) ----
cmd = ["python3", "-u", f"{CODE}/kaggle/datagen_lc0.py", "--session", str(SESSION), "--num-sessions", str(NUM_SESSIONS),
       "--target", str(TARGET), "--out", WORK, "--tmp", f"{TMP}/data", "--restore-root", "/kaggle/input",
       "--converter", f"{CODE}/build/lc0_convert", "--syzygy", TB, "--downloads", "3",
       "--start", str(START), "--deadline-h", str(DEADLINE_H)]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=CODE)
for line in p.stdout:
    print(line, end="")
assert p.wait() == 0, "datagen failed (see the log above); shards written so far are in the output"


In [ ]:
# ---- Report ----
rep = json.load(open(f"{WORK}/bulwark_data/s{SESSION}/report.json"))
print(json.dumps(rep, indent=1))
for split in ("train", "val", "test"):
    files = glob.glob(f"{WORK}/bulwark_data/s{SESSION}/training/{split}/*.bin")
    print(split, len(files), "files", round(sum(os.path.getsize(f) for f in files) / 1e9, 2), "GB (this round)")
print("\nDONE" if rep["done"] else "\nNOT DONE: add this version's output as an input and run again (it continues).")
